# Spam Detection Neural Network

This notebook builds and trains a deep learning model to classify SMS messages as spam or ham (legitimate).
It combines a 1D CNN with a Bidirectional LSTM layer for text classification.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import pickle
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Embedding, Conv1D, MaxPooling1D
from tensorflow.keras.layers import Bidirectional, LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split
import tkinter as tk
from tkinter import messagebox

## 2.  Data

In [ ]:
# Load the spam dataset
data = pd.read_csv("C:/Users/miami/Downloads/archive/spam.csv", encoding="latin-1")[["v1", "v2"]]
data.columns = ["label", "text"]
data["Label"] = data["label"].map({"ham": 0, "spam": 1})

print(f"Dataset shape: {data.shape}")
print(f"\nClass distribution:")
print(data["Label"].value_counts())
print(f"\nSample messages:")
print(data.head())

## 3. Tokenization and Padding

In [ ]:
texts = data["text"].values
labels = data["Label"].values

vocab_size = 1000
max_len = 120

# Create tokenizer and fit on texts
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)

# Convert text to sequences and pad
sequences = tokenizer.texts_to_sequences(texts)
padded = pad_sequences(sequences, maxlen=max_len)

print(f"Vocabulary size: {vocab_size}")
print(f"Max sequence length: {max_len}")
print(f"Padded sequences shape: {padded.shape}")
print(f"\nExample sequence (first 20 tokens): {padded[0][:20]}")

## 4. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    padded, labels, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
print(f"Training spam/ham ratio: {y_train.sum()} / {len(y_train) - y_train.sum()}")
print(f"Test spam/ham ratio: {y_test.sum()} / {len(y_test) - y_test.sum()}")

## 5. Build Neural Network Model

Architecture:
- **Embedding**: Converts token indices to 64-dimensional vectors
- **Conv1D**: uses 128 filters to detect local spam patterns
- **MaxPooling1D**: Downsamples to retain strongest activations
- **Bidirectional LSTM**: Captures context from both directions
- **Dense layers**: Fully connected with dropout for regularization
- **Sigmoid output**: Binary classification 

In [ ]:
embedding_dim = 64

model = Sequential([
    Embedding(vocab_size, embedding_dim),
    Conv1D(128, 5, activation="relu"),
    MaxPooling1D(pool_size=4),
    Bidirectional(LSTM(64)),
    Dense(64, activation="relu"),
    Dropout(0.5),
    Dense(1, activation="sigmoid")
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

print(model.summary())

## 6. Train the Model

In [ ]:
history = model.fit(
    X_train, y_train, 
    epochs=5, 
    batch_size=32, 
    validation_split=0.2
)

print(f"\nTraining complete!")
print(f"Final training accuracy: {history.history['accuracy'][-1]:.4f}")
print(f"Final validation accuracy: {history.history['val_accuracy'][-1]:.4f}")

## 7. Evaluate on Test Set

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test)

print(f"\nTest Set Performance:")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Loss: {test_loss:.4f}")

## 8. Save Model and Tokenizer

In [ ]:
# Save model
model.save("C:/Users/miami/spam_model.h5")
print("✓ Model saved to spam_model.h5")

# Save tokenizer
with open("C:/Users/miami/tokenizer.pkl", "wb") as t:
    pickle.dump(tokenizer, t)
print("✓ Tokenizer saved to tokenizer.pkl")

## 9. Make Predictions on New Messages

In [ ]:
def predict_message(text):
    """Predict whether a message is spam or ham."""
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_len)
    prediction = model.predict(padded, verbose=0)[0][0]
    label = "SPAM" if prediction >= 0.5 else "HAM"
    return prediction, label

# Test on sample messages
test_messages = [
    "Hey, how are you doing?",
    "Free prize! Claim now!",
    "Meeting tomorrow at 2pm",
    "Congratulations! You won a lottery"
]

print("\nPredictions on test messages:\n")
for msg in test_messages:
    pred, label = predict_message(msg)
    print(f"Message: {msg}")
    print(f"Confidence: {pred:.4f} | Label: {label}\n")

## 10.  Tkinter GUI

The GUI allows users to type messages and see instant predictions with color-coded results.

In [ ]:
# Load trained model and tokenizer
model = load_model("C:/Users/miami/spam_model.h5")
with open("C:/Users/miami/tokenizer.pkl", "rb") as t:
    tokenizer = pickle.load(t)

max_len = 120

def predict_spam():
    """Predict spam/ham for message in text box."""
    text = entry.get("1.0", tk.END).strip()
    if not text:
        messagebox.showwarning("Warning", "Enter a message")
        return
    
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_len)
    prediction = model.predict(padded, verbose=0)[0][0]
    
    if prediction >= 0.5:
        result = "The message is SPAM"
        color = "Red"
    else:
        result = "The message is NOT SPAM"
        color = "Green"
    
    result_label.config(
        text=f"{result}\nConfidence: {prediction:.4f}",
        fg=color
    )

# Create GUI window
window = tk.Tk()
window.title("Spam Detection")
window.geometry("600x600")

# Title
title = tk.Label(window, text="Spam Detection", font=("Arial", 20, "bold"))
title.pack(pady=10)

# Instructions
instructions = tk.Label(window, text="Enter a message to check if it's spam:", font=("Arial", 12))
instructions.pack(pady=5)

# Text entry box
entry = tk.Text(window, height=10, width=50, font=("Arial", 11))
entry.pack(pady=10)

# Prediction button
predict_btn = tk.Button(
    window, text="Check your message", command=predict_spam, 
    font=("Arial", 15), bg="#4CAF50", fg="white", padx=10, pady=10
)
predict_btn.pack(pady=10)

# Result label
result_label = tk.Label(window, text="", font=("Arial", 15))
result_label.pack(pady=10)

# Start GUI
window.mainloop()